# DATA PREPROCESSING

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../1-Data/heart.csv")
print('Shape',df.shape)

Shape (918, 12)


# TASK
- Remove the RestingBP outliers
- Encoding Categorical Features
- Split Train/Test
- Handle missing values (choresterol)
- Feature Scaling

# 1. Remove the RestingBp outliers
Since only 1 value RestingBp is equal 0 --> We can delete it without interupting anything else

In [3]:
df_model = df.copy()
df_model = df_model[df_model["RestingBP"] > 0]

print("Shape after removing RestingBP outliers:")
print(df_model.shape)

Shape after removing RestingBP outliers:
(917, 12)


# 2. Encoding Categorical Features
## Binary Features 
Only 2 categories --> Label Encoding (0/1):
- Sex: M -> 1, F -> 0
- Fasting B/S: already 0/1
- ExerciseAngina: Y -> 1, N -> 0

## Multi-category features 
I use one-hot encoding
- ChestPainType: 4 values → 3 dummy columns (drop_first=True)
- RestingECG: 3 values → 2 dummy columns
- ST_Slope: 3 values → 2 dummy columns

**Why drop one column:**  Dropping one dummy column avoids the Dummy Variable Trap (perfect multicollinearity). When all dummy variables for a categorical feature are included along with an intercept term, one column becomes perfectly predictable from the others because the dummy variables always sum to 1. This makes the feature matrix non–full rank, causing unstable or non-unique coefficient estimates in Logistic Regression. By dropping one category, we create a reference category that the model infers implicitly, eliminating the redundancy.

**Note:** One-hot encoding is applied before splitting because all categories are known from the full dataset. For this specific case, this does not introduce data leakage since we are not learning any statistics (only creating binary columns). In production systems, encoding should be fitted on training data only.

In [4]:
df_model['Sex'] = df_model['Sex'].map({'M': 1, 'F': 0})
df_model['ExerciseAngina'] = df_model['ExerciseAngina'].map({'Y': 1, 'N': 0})

df_model = pd.get_dummies(df_model, columns=['ChestPainType', 'RestingECG', 'ST_Slope'], drop_first=True, dtype=int)

In [5]:
print(df_model.shape)
df_model.head()

(917, 16)


,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,HeartDisease,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up
0,40,1,140,289,0,172,0,0.0,0,1,0,0,1,0,0,1
1,49,0,160,180,0,156,0,1.0,1,0,1,0,1,0,1,0
2,37,1,130,283,0,98,0,0.0,0,1,0,0,0,1,0,1
3,48,0,138,214,0,108,1,1.5,1,0,0,0,1,0,1,0
4,54,1,150,195,0,122,0,0.0,0,0,1,0,1,0,0,1


# 3. Split Train/Test  
I decided to split to 80 train - 20 test

In [6]:
from sklearn.model_selection import train_test_split

X = df_model.drop('HeartDisease', axis=1)
y = df_model['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 80% train, 20% test
    random_state=42,      # reproducibility
    stratify=y            # preserve class distribution -> ensure both train and test set have same ratio
)
df_train = pd.concat([X_train, y_train], axis=1)
df_test = pd.concat([X_test, y_test], axis=1)

print(f"Train: {df_train.shape}, Test: {df_test.shape}")
df_train.to_csv("../1-Data/train_data.csv", index=False)
df_test.to_csv("../1-Data/test_data.csv", index=False)

Train: (733, 16), Test: (184, 16)


In [7]:
df = pd.read_csv("../1-Data/train_data.csv")
df.head()

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up,HeartDisease
0,49,1,128,212,0,96,1,0.0,0,0,0,1,0,1,0,1
1,68,1,134,254,1,151,1,0.0,0,1,0,1,0,0,1,0
2,59,1,126,218,1,134,0,2.2,0,1,0,1,0,1,0,1
3,67,1,125,254,1,163,0,0.2,0,0,0,1,0,1,0,1
4,52,1,160,331,0,94,1,2.5,0,0,0,1,0,1,0,1


# 4. Handle missing value on train_data
Using median to replace zero of non-zero Cholesterol values  
Why median (not mean)? Cholesterol distributions are right-skewed; median is more robust to outliers.   
---See more in EDA file---

In [8]:
# Replace 0 Cholesterol with median of non-zero values
median_chol = df[df['Cholesterol'] > 0]['Cholesterol'].median()
df['Cholesterol'] = df['Cholesterol'].replace(0, median_chol)

# Check RestingBP zeros
print(f"RestingBP zeros: {(df['RestingBP'] == 0).sum()}")

RestingBP zeros: 0


# 5. Feature scaling
Note: only on training data - fitting on test causes data leakage.   
Role: 
- Centre data to mean=0, std=1
- Required for Logistic Regression, Neural Networks and PCA

In [9]:
from sklearn.preprocessing import StandardScaler

X_train = df.drop('HeartDisease', axis=1)
y_train = df['HeartDisease']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [10]:
df_train_processed = pd.DataFrame(X_train_scaled, columns=X_train.columns)
df_train_processed['HeartDisease'] = y_train.values

df_train_processed.to_csv("../1-Data/train_processed.csv", index=False)

In [11]:
df = pd.read_csv("../1-Data/train_processed.csv")
df.head()

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up,HeartDisease
0,-0.504752,0.509363,-0.271661,-0.606403,-0.551607,-1.590143,1.218501,-0.813322,-0.47729,-0.522082,-0.23383,0.813715,-0.488032,0.987796,-0.870514,1
1,1.505450,0.509363,0.055941,0.171947,1.812884,0.557626,1.218501,-0.813322,-0.47729,1.915409,-0.23383,0.813715,-0.488032,-1.012355,1.148747,0
2,0.553249,0.509363,-0.380862,-0.495210,1.812884,-0.106230,-0.820680,1.244032,-0.47729,1.915409,-0.23383,0.813715,-0.488032,0.987796,-0.870514,1
3,1.399650,0.509363,-0.435463,0.171947,1.812884,1.026231,-0.820680,-0.626290,-0.47729,-0.522082,-0.23383,0.813715,-0.488032,0.987796,-0.870514,1
4,-0.187351,0.509363,1.475552,1.598922,-0.551607,-1.668244,1.218501,1.524581,-0.47729,-0.522082,-0.23383,0.813715,-0.488032,0.987796,-0.870514,1


# 6. Do the same for test data  
- Use median_chol in train. won't calculate median again to prevent data leakage  
- feature scaling: only use fit

In [12]:
df_test = pd.read_csv("../1-Data/test_data.csv")

df_test['Cholesterol'] = df_test['Cholesterol'].replace(0, median_chol)

X_test_unscaled = df_test.drop('HeartDisease', axis=1)
y_test = df_test['HeartDisease']

X_test_scaled = scaler.transform(X_test_unscaled)

df_test_processed = pd.DataFrame(X_test_scaled, columns=X_test_unscaled.columns)
df_test_processed['HeartDisease'] = y_test.values

df_test_processed.to_csv("../1-Data/test_processed.csv", index=False)

In [13]:
df = pd.read_csv("../1-Data/test_processed.csv")
df.head()

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ST_Slope_Flat,ST_Slope_Up,HeartDisease
0,0.341649,0.509363,-0.271661,-0.291357,-0.551607,0.518576,-0.820680,-0.439258,-0.47729,1.915409,-0.23383,-1.228932,-0.488032,0.987796,-0.870514,1
1,0.130049,0.509363,0.383544,-0.513742,-0.551607,-1.004388,1.218501,4.423581,-0.47729,-0.522082,-0.23383,0.813715,-0.488032,-1.012355,-0.870514,1
2,0.870650,0.509363,1.038749,-1.699799,-0.551607,-1.551093,1.218501,0.682936,-0.47729,-0.522082,-0.23383,-1.228932,2.049044,-1.012355,1.148747,1
3,0.024249,0.509363,-0.599264,0.764975,-0.551607,-0.809136,1.218501,2.179194,-0.47729,-0.522082,-0.23383,-1.228932,-0.488032,0.987796,-0.870514,1
4,-1.562753,-1.963238,0.274343,-0.458146,-0.551607,0.596677,-0.820680,-0.813322,-0.47729,1.915409,-0.23383,0.813715,-0.488032,0.987796,-0.870514,0


In [15]:
import pandas as pd
df_test = pd.read_csv("../1-Data/test_data.csv")

# Save true/false when modify cholesterol
df_test['Cholesterol_Modified'] = (df_test['Cholesterol'] == 0).astype(int)
df_test['Cholesterol_Modified'] = df_test['Cholesterol'] == 0

# Replace 0 by median_cho
df_test['Cholesterol'] = df_test['Cholesterol'].replace(0, median_chol)

df_test.to_csv("../1-Data/test_for_ea.csv", index=False)